In [24]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# 1. LOAD DATA
# -----------------------------
df = pd.read_csv("netflix_movies_detailed_up_to_2025.csv")

# Select only useful columns
df = df[['title', 'genres', 'description', 'cast', 'director']]

# Fill missing values
df.fillna('', inplace=True)

# -----------------------------
# 2. CLEAN TEXT
# -----------------------------
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for col in ['genres', 'description', 'cast', 'director']:
    df[col] = df[col].apply(clean)

# -----------------------------
# 3. TF-IDF VECTORIZATION (SEPARATE)
# -----------------------------

# Genres (HIGH priority)
tfidf_genre = TfidfVectorizer(stop_words='english')
genre_matrix = tfidf_genre.fit_transform(df['genres'])

# Description (MEDIUM)
tfidf_desc = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))
desc_matrix = tfidf_desc.fit_transform(df['description'])

# Cast (LOW-MEDIUM)
tfidf_cast = TfidfVectorizer(stop_words='english')
cast_matrix = tfidf_cast.fit_transform(df['cast'])

# Director (LOW)
tfidf_director = TfidfVectorizer(stop_words='english')
director_matrix = tfidf_director.fit_transform(df['director'])

# -----------------------------
# 4. SIMILARITY MATRICES
# -----------------------------
genre_sim = cosine_similarity(genre_matrix)
desc_sim = cosine_similarity(desc_matrix)
cast_sim = cosine_similarity(cast_matrix)
director_sim = cosine_similarity(director_matrix)

# -----------------------------
# 5. WEIGHTED FINAL SIMILARITY
# -----------------------------
# Adjust weights here if needed
final_sim = (
    0.5 * genre_sim +      # highest weight
    0.3 * desc_sim +
    0.15 * cast_sim +
    0.05 * director_sim
)

# -----------------------------
# 6. CREATE TITLE INDEX (FAST LOOKUP)
# -----------------------------
df['title_lower'] = df['title'].str.lower()
indices = pd.Series(df.index, index=df['title_lower']).drop_duplicates()

# -----------------------------
# 7. RECOMMENDATION FUNCTION
# -----------------------------
def recommend(title, top_n=5):
    title = title.lower()

    if title not in indices:
        return ["Movie not found"]

    idx = indices[title]

    scores = list(enumerate(final_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    results = [df.iloc[i[0]]['title'] for i in scores]

    return results

# -----------------------------
# 8. TEST
# -----------------------------
print(recommend("Inception",top_n=10))
import pandas as pd

df.to_pickle("movies.pkl")
pd.to_pickle(final_sim, "similarity.pkl")
pd.to_pickle(indices, "indices.pkl")

['Venom: Let There Be Carnage', 'Mad Max: Fury Road', 'The Creator', 'Venom: The Last Dance', 'Guardians of the Galaxy', 'The Call Up', 'Battle of the Damned', 'Ant-Man', 'The Amazing Spider-Man 2', 'After Earth']
